# Cleaning_CYRV_order_items_dataset.csv

## Conclusion

The `CYRV_order_items_dataset` was validated and prepared for downstream analysis without removing any rows or imputing any values.

The dataset contains 112,650 order-item records with no missing values, duplicate rows, duplicate `(order_id, order_item_id)` combinations, or negative price/freight values. Product, seller, and order relationships were validated against the cleaned datasets.

Several unusual but potentially valid patterns were retained, including zero freight values, high price and freight values, repeated products within orders, and multi-seller orders. Four records belonging to three orders were identified with extreme `shipping_limit_date` values in 2020, more than 1,000 days after purchase. These records were retained because the correct shipping-limit dates cannot be reliably reconstructed.

The cleaned dataset is structurally consistent and ready for downstream analysis, with the documented shipping-limit anomalies taken into account.

## Cleaning Summary

- Dataset shape: 112,650 rows × 7 columns.
- No missing values were found.
- No fully duplicated rows were found.
- The combination of `order_id` and `order_item_id` is unique for every row.
- `order_item_id` sequences within orders were validated and contain no gaps.
- `shipping_limit_date` was converted to datetime successfully with no values lost during conversion.
- No `shipping_limit_date` occurs before its corresponding order purchase timestamp.
- `price` contains no zero or negative values.
- `freight_value` contains no negative values. 383 zero-freight records were retained as plausible free-shipping cases.
- High price and freight values were inspected and retained because there was no evidence that they were invalid.
- All 32,951 product IDs in the order-items dataset exist in the cleaned products dataset, and every cleaned product appears in at least one order item.
- All 3,095 seller IDs in the order-items dataset exist in the cleaned sellers dataset, and every cleaned seller appears in at least one order item.
- All 98,666 order IDs represented in the order-items dataset exist in the cleaned orders dataset.
- 775 orders have no corresponding order-item records. Most are unavailable or canceled orders; one shipped order without order-item records was identified and retained as a documented source-data inconsistency.
- Repeated products within the same order were retained because they represent multiple units rather than duplicate records. These repetitions occur in 6,968 orders, affecting 17,313 rows.
- Within a single order, the same product is never associated with multiple sellers.
- 1,278 orders (1.3% of orders represented in `order_items`) contain multiple sellers, with a maximum of 5 sellers in one order.
- 1,225 products (3.72% of products) are sold by multiple sellers across different orders, with a maximum of 8 sellers for one product.
- Four order-item records belonging to three orders have extreme `shipping_limit_date` values in 2020, approximately 1,052–1,056 days after purchase. All four records belong to the same seller. The dates were retained because no reliable replacement values are available.
- No rows were removed and no values were imputed or manually corrected.
- The cleaned dataset was saved as `CYRV_order_items_cleaned.csv` and successfully reloaded with 112,650 rows, no missing values, and no duplicate rows.

In [1]:
import pandas as pd

order_items = pd.read_csv("/Users/yuliiapotrymai/Desktop/SpikupCapstone2026_CYRV/data/cyrv/CYRV_order_items_dataset.csv")

order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [2]:
print("Shape:", order_items.shape)

print("\nData types:")
print(order_items.dtypes)

Shape: (112650, 7)

Data types:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object


In [3]:
print("Missing values:")
print(order_items.isna().sum())

print("\nDuplicate rows:")
print(order_items.duplicated().sum())

print("\nUnique values:")
print("Unique order IDs:", order_items["order_id"].nunique())
print("Unique product IDs:", order_items["product_id"].nunique())
print("Unique seller IDs:", order_items["seller_id"].nunique())

print("\nOrder item ID range:")
print("Min:", order_items["order_item_id"].min())
print("Max:", order_items["order_item_id"].max())

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Duplicate rows:
0

Unique values:
Unique order IDs: 98666
Unique product IDs: 32951
Unique seller IDs: 3095

Order item ID range:
Min: 1
Max: 21


In [4]:
print(
    "Duplicate order_id + order_item_id combinations:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

items_per_order = (
    order_items.groupby("order_id")["order_item_id"].count()
)

print("\nItems per order:")
print(items_per_order.describe())


Duplicate order_id + order_item_id combinations: 0

Items per order:
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: order_item_id, dtype: float64


In [5]:
item_sequence_check = (
    order_items.groupby("order_id")["order_item_id"]
    .apply(
        lambda x: sorted(x.tolist())
        == list(range(1, len(x) + 1))
    )
)

print("Orders with valid item sequence:", item_sequence_check.sum())
print("Orders with invalid item sequence:", (~item_sequence_check).sum())


Orders with valid item sequence: 98666
Orders with invalid item sequence: 0


In [6]:
print("Missing before conversion:")
print(order_items["shipping_limit_date"].isna().sum())

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

print("\nData type after conversion:")
print(order_items["shipping_limit_date"].dtype)

print("\nMissing after conversion:")
print(order_items["shipping_limit_date"].isna().sum())

Missing before conversion:
0

Data type after conversion:
datetime64[us]

Missing after conversion:
0


In [7]:
print("Price:")
print(order_items["price"].describe())
print("Negative prices:", (order_items["price"] < 0).sum())
print("Zero prices:", (order_items["price"] == 0).sum())

print("\nFreight value:")
print(order_items["freight_value"].describe())
print("Negative freight values:", (order_items["freight_value"] < 0).sum())
print("Zero freight values:", (order_items["freight_value"] == 0).sum())

Price:
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64
Negative prices: 0
Zero prices: 0

Freight value:
count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64
Negative freight values: 0
Zero freight values: 383


In [8]:
products = pd.read_csv("../CYRV_products_cleaned.csv")

order_item_product_ids = set(order_items["product_id"])
product_ids = set(products["product_id"])

print("Product IDs in order items:", len(order_item_product_ids))
print("Product IDs in products:", len(product_ids))

print(
    "Product IDs missing from products:",
    len(order_item_product_ids - product_ids)
)

print(
    "Products without order items:",
    len(product_ids - order_item_product_ids)
)

Product IDs in order items: 32951
Product IDs in products: 32951
Product IDs missing from products: 0
Products without order items: 0


In [9]:
sellers = pd.read_csv("../CYRV_sellers_cleaned.csv")

order_item_seller_ids = set(order_items["seller_id"])
seller_ids = set(sellers["seller_id"])

print("Seller IDs in order items:", len(order_item_seller_ids))
print("Seller IDs in sellers:", len(seller_ids))

print(
    "Seller IDs missing from sellers:",
    len(order_item_seller_ids - seller_ids)
)

print(
    "Sellers without order items:",
    len(seller_ids - order_item_seller_ids)
)

Seller IDs in order items: 3095
Seller IDs in sellers: 3095
Seller IDs missing from sellers: 0
Sellers without order items: 0


In [10]:
orders = pd.read_csv("../CYRV_orders_cleaned.csv")

order_item_order_ids = set(order_items["order_id"])
order_ids = set(orders["order_id"])

print("Order IDs in order items:", len(order_item_order_ids))
print("Order IDs in orders:", len(order_ids))

print(
    "Order item IDs missing from orders:",
    len(order_item_order_ids - order_ids)
)

print(
    "Orders without order items:",
    len(order_ids - order_item_order_ids)
)

Order IDs in order items: 98666
Order IDs in orders: 99441
Order item IDs missing from orders: 0
Orders without order items: 775


In [11]:
orders_without_items = orders[
    ~orders["order_id"].isin(order_item_order_ids)
].copy()

print("Orders without order items:", len(orders_without_items))

print("\nOrder statuses:")
print(orders_without_items["order_status"].value_counts())

Orders without order items: 775

Order statuses:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [12]:
orders_without_items[
    orders_without_items["order_status"] == "shipped"
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
23254,a68ce1686d536ca72bd2dadc4b8671e5,d7bed5fac093a4136216072abaf599d5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,NaN,2016-12-01


In [13]:
print("Highest prices:")
display(
    order_items[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "price",
            "freight_value"
        ]
    ]
    .sort_values("price", ascending=False)
    .head(10)
)

print("\nHighest freight values:")
display(
    order_items[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "price",
            "freight_value"
        ]
    ]
    .sort_values("freight_value", ascending=False)
    .head(10)
)

Highest prices:


,order_id,order_item_id,product_id,seller_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,1,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,1,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,1,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,1,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,1,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,3999.90,17.01



Highest freight values:


,order_id,order_item_id,product_id,seller_id,price,freight_value
73486,a77e1550db865202c56b19ddc6dc4d53,1,ec31d2a17b299511e7c8627be9337b9b,257e61d3251fb5efb9daadddbc2cf7ca,979.00,409.68
28044,3fde74c28a3d5d618c00f26d51baafa0,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2338.08,375.28
3303,076d1555fb53a89b0ef4d529e527a0f6,1,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2338.08,375.28
69797,9f49bd16053df810384e793386312674,1,256a9c364b75753b97bee410c9491ad8,5c030029b5916fed0986310385ec9009,1149.00,339.59
16731,264a7e199467906c0727394df82d1a6a,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,1050.00,338.30
87936,c7a07ddd52bbe18b61da49a8d89853d3,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,1050.00,322.10
5037,0b6230647ed16f4b3e70282dc4b5b87f,1,46e24ce614899e36617e37ea1e4aa6ff,17f51e7198701186712e53a39c564617,1050.00,321.88
3584,0822bcde10bb5d023755a71bc8f7797f,1,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,990.00,321.46
29787,43bdbd9dc0931d72befdf4765af6c442,1,7e53e051875b2a0c9f22acd8a9a29a20,eeb6de78f79159600292e314a77cbd18,3089.00,317.47
48320,6ddfbf514959b49b6410c01ad93054bb,1,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,1045.00,314.40


In [14]:
order_dates = orders[
    ["order_id", "order_purchase_timestamp"]
].copy()

order_dates["order_purchase_timestamp"] = pd.to_datetime(
    order_dates["order_purchase_timestamp"],
    errors="coerce"
)

shipping_check = order_items.merge(
    order_dates,
    on="order_id",
    how="left"
)

shipping_before_purchase = (
    shipping_check["shipping_limit_date"]
    < shipping_check["order_purchase_timestamp"]
)

print(
    "Shipping limit before purchase:",
    shipping_before_purchase.sum()
)

Shipping limit before purchase: 0


In [15]:
repeated_products = order_items.duplicated(
    subset=["order_id", "product_id"],
    keep=False
)

print(
    "Rows with repeated product within the same order:",
    repeated_products.sum()
)

print(
    "Orders with repeated product:",
    order_items.loc[repeated_products, "order_id"].nunique()
)

Rows with repeated product within the same order: 17313
Orders with repeated product: 6968


In [16]:
order_items.loc[
    repeated_products,
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].sort_values(
    ["order_id", "product_id", "order_item_id"]
).head(20)

,order_id,order_item_id,product_id,seller_id,price,freight_value
13,0008288aa423d2a3f00fcb17cd7d8719,1,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,49.90,13.37
14,0008288aa423d2a3f00fcb17cd7d8719,2,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,49.90,13.37
32,00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,21.33,15.10
33,00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,21.33,15.10
34,00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,21.33,15.10
42,001ab0a7578dd66cd4b0a71f5b6e1e41,1,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,24.89,17.63
43,001ab0a7578dd66cd4b0a71f5b6e1e41,2,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,24.89,17.63
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,24.89,17.63
48,001d8f0e34a38c37f7dba2a37d4eba8b,1,e67307ff0f15ade43fcb6e670be7a74c,f4aba7c0bca51484c30ab7bdc34bcdd1,18.99,7.78
49,001d8f0e34a38c37f7dba2a37d4eba8b,2,e67307ff0f15ade43fcb6e670be7a74c,f4aba7c0bca51484c30ab7bdc34bcdd1,18.99,7.78


In [17]:
product_sellers_per_order = (
    order_items.groupby(["order_id", "product_id"])["seller_id"]
    .nunique()
)

print(
    "Order-product combinations with multiple sellers:",
    (product_sellers_per_order > 1).sum()
)

print(
    "Maximum sellers for the same product in one order:",
    product_sellers_per_order.max()
)

Order-product combinations with multiple sellers: 0
Maximum sellers for the same product in one order: 1


In [19]:
sellers_per_order = (
    order_items.groupby("order_id")["seller_id"]
    .nunique()
)

print(
    "Orders with multiple sellers:",
    (sellers_per_order > 1).sum()
)

print(
    "Percentage of orders with multiple sellers:",
    round((sellers_per_order > 1).mean() * 100, 2),
    "%"
)

print(
    "Maximum sellers in one order:",
    sellers_per_order.max()
)

Orders with multiple sellers: 1278
Percentage of orders with multiple sellers: 1.3 %
Maximum sellers in one order: 5


In [20]:
sellers_per_product = (
    order_items.groupby("product_id")["seller_id"]
    .nunique()
)

print(
    "Products sold by multiple sellers:",
    (sellers_per_product > 1).sum()
)

print(
    "Percentage of products sold by multiple sellers:",
    round((sellers_per_product > 1).mean() * 100, 2),
    "%"
)

print(
    "Maximum sellers for one product:",
    sellers_per_product.max()
)

Products sold by multiple sellers: 1225
Percentage of products sold by multiple sellers: 3.72 %
Maximum sellers for one product: 8


In [21]:
print(
    "Shipping limit date range:",
    order_items["shipping_limit_date"].min(),
    "to",
    order_items["shipping_limit_date"].max()
)

Shipping limit date range: 2016-09-19 00:15:34 to 2020-04-09 22:35:08


In [22]:
future_shipping_limits = order_items[
    order_items["shipping_limit_date"] >= "2019-01-01"
].copy()

print("Rows with shipping limit date in 2019 or later:", len(future_shipping_limits))

display(
    future_shipping_limits[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "shipping_limit_date",
            "price",
            "freight_value"
        ]
    ].sort_values("shipping_limit_date")
)

Rows with shipping limit date in 2019 or later: 4


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,75.99,14.70
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,69.99,14.66
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44


In [23]:
future_order_ids = future_shipping_limits["order_id"].unique()

orders[
    orders["order_id"].isin(future_order_ids)
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
36034,13bdf405f961a6deec817d817f5c6624,canceled,2017-03-16 02:30:51,2017-03-16 02:30:51,NaN,NaN,2017-08-08
55368,9c94a4ea2f7876660fa6f1b59b69c8e6,shipped,2017-03-14 19:23:22,2017-03-14 19:23:22,2017-03-16 14:31:15,NaN,2017-08-04
86444,c2bb89b5c1dd978d507284be78a04cb2,delivered,2017-05-23 22:28:36,2017-05-24 22:35:08,2017-05-29 02:03:28,2017-06-09 13:35:54,2017-10-11


In [24]:
shipping_check["shipping_limit_days"] = (
    shipping_check["shipping_limit_date"]
    - shipping_check["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print(shipping_check["shipping_limit_days"].describe())

print("\nLongest shipping limit intervals:")
display(
    shipping_check[
        [
            "order_id",
            "order_item_id",
            "shipping_limit_date",
            "order_purchase_timestamp",
            "shipping_limit_days"
        ]
    ]
    .sort_values("shipping_limit_days", ascending=False)
    .head(10)
)

count    112650.000000
mean          6.643793
std           7.069132
min           2.004236
25%           5.006863
50%           6.011991
75%           7.187535
max        1056.041667
Name: shipping_limit_days, dtype: float64

Longest shipping limit intervals:


,order_id,order_item_id,shipping_limit_date,order_purchase_timestamp,shipping_limit_days
8643,13bdf405f961a6deec817d817f5c6624,1,2020-02-05 03:30:51,2017-03-16 02:30:51,1056.041667
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,2020-02-03 20:23:22,2017-03-14 19:23:22,1056.041667
85730,c2bb89b5c1dd978d507284be78a04cb2,2,2020-04-09 22:35:08,2017-05-23 22:28:36,1052.004537
85729,c2bb89b5c1dd978d507284be78a04cb2,1,2020-04-09 22:35:08,2017-05-23 22:28:36,1052.004537
46557,69d126e78947276280838ee9361f5505,1,2017-08-25 15:32:11,2017-03-30 15:23:23,148.006111
28492,40dc2ba6f322a17626aac6244332828c,1,2018-01-10 22:49:21,2017-10-05 21:39:05,97.048796
87724,c72727d29cde4cf870d569bf65edabfd,1,2017-05-12 17:01:15,2017-02-07 18:01:15,93.958333
24130,36f66ebfbd144aa6fb39338bd87b0bf9,1,2018-05-29 03:36:33,2018-02-26 00:11:02,92.142720
29936,440b5d8129165b7e04b3c37e3a5e93d7,1,2017-08-03 09:31:38,2017-05-21 08:01:31,74.062581
20220,2e5dc86c8c4aa663549caf5e31de840d,1,2017-04-13 10:56:48,2017-02-03 00:04:49,69.452766


In [25]:
thresholds = [30, 60, 90, 180, 365]

for days in thresholds:
    count = (shipping_check["shipping_limit_days"] > days).sum()
    print(f"More than {days} days: {count}")

More than 30 days: 239
More than 60 days: 12
More than 90 days: 8
More than 180 days: 4
More than 365 days: 4


In [26]:
print("Shape:", order_items.shape)
print("Duplicate rows:", order_items.duplicated().sum())

print(
    "Duplicate order_id + order_item_id:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print("\nMissing values:")
print(order_items.isna().sum())

print("\nData types:")
print(order_items.dtypes)

print("\nNegative values:")
print("Negative prices:", (order_items["price"] < 0).sum())
print("Negative freight values:", (order_items["freight_value"] < 0).sum())

Shape: (112650, 7)
Duplicate rows: 0
Duplicate order_id + order_item_id: 0

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Data types:
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

Negative values:
Negative prices: 0
Negative freight values: 0


In [27]:
output_path = "../CYRV_order_items_cleaned.csv"

order_items.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")

Saved cleaned dataset to: ../CYRV_order_items_cleaned.csv


In [28]:
cleaned_order_items = pd.read_csv(
    "../CYRV_order_items_cleaned.csv",
    parse_dates=["shipping_limit_date"]
)

print("Shape:", cleaned_order_items.shape)
print("Duplicate rows:", cleaned_order_items.duplicated().sum())

print("\nMissing values:")
print(cleaned_order_items.isna().sum())

print("\nData types:")
print(cleaned_order_items.dtypes)

Shape: (112650, 7)
Duplicate rows: 0

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Data types:
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object
